# Perceptron e Adaline — Porta Lógica AND
**Disciplina:** Redes Neurais  
**Restrições:** apenas `numpy` e `matplotlib`

In [ ]:
# ── Imports e dados ───────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# Porta AND: entradas binárias, saída bipolar (-1 / +1)
X_raw = np.array([[0, 0],
                   [0, 1],
                   [1, 0],
                   [1, 1]], dtype=float)
d = np.array([-1, -1, -1, 1], dtype=float)

# Bias: coluna x0 = 1 inserida na posição 0
X = np.hstack([np.ones((4, 1)), X_raw])   # shape (4, 3)

ETA        = 0.1
MAX_EPOCHS = 100

def sign(v):
    return 1.0 if v >= 0 else -1.0

print("Dados de treinamento (com bias):")
print(f"{'x0':>4} {'x1':>4} {'x2':>4} {'d':>4}")
for i in range(len(X)):
    print(f"{X[i,0]:>4.0f} {X[i,1]:>4.0f} {X[i,2]:>4.0f} {d[i]:>4.0f}")

## Parte 1 — Perceptron (Regra de Rosenblatt)
$$w \leftarrow w + \eta \cdot (d - y) \cdot x \qquad y = \text{sign}(w \cdot x)$$

**Inicialização:** pesos zerados `w = [0, 0, 0]`

In [ ]:
# ── Parte 1: Perceptron ───────────────────────────────────────────────────────
w_p = np.zeros(3)       # w0 (bias), w1, w2
perceptron_errors = []

print("=" * 40)
print("PERCEPTRON — η=0.1, pesos iniciais=[0,0,0]")
print("=" * 40)

epochs_p = 0
for epoch in range(MAX_EPOCHS):
    errors = 0
    for i in range(len(X)):
        y = sign(np.dot(w_p, X[i]))
        delta = d[i] - y
        if delta != 0:
            w_p += ETA * delta * X[i]
            errors += 1
    perceptron_errors.append(errors)
    print(f"  Época {epoch+1:3d} | Erros: {errors}")
    epochs_p = epoch + 1
    if errors == 0:
        print(f"  → Convergiu na época {epochs_p}.")
        break

print()
print(f"Pesos finais : w0={w_p[0]:.4f}, w1={w_p[1]:.4f}, w2={w_p[2]:.4f}")
print(f"Épocas       : {epochs_p}")
print(f"Reta de decisão: {w_p[0]:.4f} + {w_p[1]:.4f}·x1 + {w_p[2]:.4f}·x2 = 0")

## Parte 2 — Adaline (Regra Delta / Widrow-Hoff)
$$w \leftarrow w + \eta \cdot (d - w \cdot x) \cdot x \qquad \text{(erro sobre saída linear)}$$

Parada: $|\text{SSE}_{t} - \text{SSE}_{t-1}| < 10^{-6}$ ou limite de épocas.

In [ ]:
# ── Parte 2: Adaline ──────────────────────────────────────────────────────────
w_a = np.zeros(3)       # mesma inicialização
adaline_sse = []
prev_sse = None

print("=" * 40)
print("ADALINE — η=0.1, pesos iniciais=[0,0,0]")
print("=" * 40)

epochs_a = 0
for epoch in range(MAX_EPOCHS):
    sse = 0.0
    for i in range(len(X)):
        u     = np.dot(w_a, X[i])       # saída linear (sem sign)
        error = d[i] - u
        w_a  += ETA * error * X[i]
        sse  += error ** 2
    adaline_sse.append(sse)
    print(f"  Época {epoch+1:3d} | SSE: {sse:.6f}")
    epochs_a = epoch + 1
    if prev_sse is not None and abs(prev_sse - sse) < 1e-6:
        print(f"  → Convergiu na época {epochs_a} (ΔSSE < 1e-6).")
        break
    prev_sse = sse

print()
print(f"Pesos finais : w0={w_a[0]:.4f}, w1={w_a[1]:.4f}, w2={w_a[2]:.4f}")
print(f"Épocas       : {epochs_a}")
print(f"Reta de decisão: {w_a[0]:.4f} + {w_a[1]:.4f}·x1 + {w_a[2]:.4f}·x2 = 0")

print("\nVerificação (sign aplicado apenas na predição final):")
print(f"{'x1':>4} {'x2':>4} {'d':>4} {'ŷ (Perceptron)':>16} {'ŷ (Adaline)':>12}")
for i in range(len(X)):
    yp = sign(np.dot(w_p, X[i]))
    ya = sign(np.dot(w_a, X[i]))
    print(f"{X[i,1]:>4.0f} {X[i,2]:>4.0f} {d[i]:>4.0f} {yp:>16.0f} {ya:>12.0f}")

## Parte 3 — Predição Interativa
Execute esta célula **após** as células de treino acima. Digite `sair` para encerrar.

In [ ]:
# ── Parte 3: Predição interativa ──────────────────────────────────────────────
MODEL_WEIGHTS = {"perceptron": w_p, "adaline": w_a}

def interactive_predict():
    print("Modelos disponíveis: perceptron | adaline")
    escolha = input("Escolha o modelo: ").strip().lower()
    if escolha not in MODEL_WEIGHTS:
        print("Modelo inválido. Usando perceptron.")
        escolha = "perceptron"
    w = MODEL_WEIGHTS[escolha]
    print(f"Modelo: {escolha}. Digite 'sair' para encerrar.\n")

    while True:
        x1_in = input("x1: ").strip()
        if x1_in.lower() == "sair":
            break
        x2_in = input("x2: ").strip()
        if x2_in.lower() == "sair":
            break
        try:
            x1_val = float(x1_in)
            x2_val = float(x2_in)
        except ValueError:
            print("  Entrada inválida. Use números (0 ou 1).\n")
            continue
        resultado = sign(np.dot(w, np.array([1.0, x1_val, x2_val])))
        print(f"  Saída [{escolha}]: {resultado:+.0f}\n")

    print("Encerrando predição interativa.")

interactive_predict()

## Parte 4 — Comparação

In [ ]:
# ── Gráficos ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ep_range_p = range(1, epochs_p + 1)
axes[0].plot(ep_range_p, perceptron_errors, marker='o', linewidth=2,
             color='steelblue', label='Perceptron')
axes[0].fill_between(ep_range_p, perceptron_errors, alpha=0.15, color='steelblue')
axes[0].set_title('Perceptron — Erros por Época', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Número de Erros de Classificação')
axes[0].set_xticks(list(ep_range_p))
axes[0].set_ylim(-0.2, max(perceptron_errors) + 1)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

ep_range_a = range(1, epochs_a + 1)
axes[1].plot(ep_range_a, adaline_sse, marker='s', linewidth=2,
             color='coral', label='Adaline')
axes[1].fill_between(ep_range_a, adaline_sse, alpha=0.15, color='coral')
axes[1].set_title('Adaline — SSE por Época', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Soma dos Erros Quadráticos (SSE)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle('Curvas de Aprendizado — Porta AND', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico salvo em curvas_aprendizado.png")

In [ ]:
# ── Tabela comparativa ────────────────────────────────────────────────────────
sep = "=" * 62
lin = "-" * 62

print(sep)
print(f"{'TABELA COMPARATIVA':^62}")
print(sep)
print(f"{'Parâmetro':<24} {'Perceptron':>18} {'Adaline':>18}")
print(lin)
print(f"{'w0 (bias)':24} {w_p[0]:>18.4f} {w_a[0]:>18.4f}")
print(f"{'w1':24} {w_p[1]:>18.4f} {w_a[1]:>18.4f}")
print(f"{'w2':24} {w_p[2]:>18.4f} {w_a[2]:>18.4f}")
print(f"{'Épocas até parada':24} {epochs_p:>18d} {epochs_a:>18d}")
print(lin)
print()
print("Equação da reta de decisão  (w0 + w1·x1 + w2·x2 = 0):")
print(f"  Perceptron : {w_p[0]:.4f} + ({w_p[1]:.4f})·x1 + ({w_p[2]:.4f})·x2 = 0")
print(f"  Adaline    : {w_a[0]:.4f} + ({w_a[1]:.4f})·x1 + ({w_a[2]:.4f})·x2 = 0")

## Análise Comparativa

O **Perceptron** utiliza a saída discreta — após a função sinal — para calcular o erro de cada exemplo. Isso significa que o peso só é atualizado quando a classificação final está **errada** (erro binário: 0 ou ±2). Consequentemente, a curva de erros por época é **irregular**: um ajuste feito para corrigir um exemplo pode desfazer a classificação correta de outro, causando oscilação entre épocas. O algoritmo percorre o espaço de pesos de forma abrupta, fazendo "saltos" até encontrar uma solução viável.

O **Adaline**, por outro lado, calcula o erro sobre a **saída linear** (contínua), antes de aplicar qualquer função de ativação. Esse erro contínuo é minimizado via **gradiente descendente** sobre o erro quadrático (SSE). O resultado é uma curva de aprendizado **monotonicamente decrescente e suave**, análoga à descida de uma encosta em direção ao mínimo global. O Adaline continua ajustando os pesos mesmo quando a classificação já está correta, refinando a posição da fronteira de decisão.

A diferença fundamental está em **onde cada modelo mede o erro**: o Perceptron só reage a erros de classificação (perspectiva binária), enquanto o Adaline mede a distância contínua entre a ativação linear e o alvo desejado. Isso confere ao Adaline maior estabilidade e tendência a encontrar fronteiras de decisão mais robustas — com maior margem — em relação ao simples Perceptron, que para na primeira solução que classifica tudo corretamente, sem garantia de otimalidade.